# 01 – Đọc số liệu và thống kê cơ bản

**Bài:** Phân bố thống kê trong phân rã phóng xạ – Nhóm 3

**Mục tiêu**
1. Đọc số liệu thô từ file Excel bằng `pandas`, xác định đúng vùng ô.
2. Kiểm tra dữ liệu (kích thước, kiểu, giá trị thiếu).
3. Tính trị trung bình, độ lệch chuẩn, độ không đảm bảo và **tránh bẫy `ddof`**.
4. Kiểm tra tính chất Poisson $\sigma^2 = \mu$ (chỉ số phân tán $D$).
5. So sánh 100 lần đo 10 s với một lần đo 100 s (chỉ số $z$).
6. Đóng gói việc đọc số liệu thành hàm để tái sử dụng.

> Ứng với **mục 2.3 bước 2** trong `00_DeBai/Poisson.pdf`.

## 1. Thư viện
| Thư viện | Dùng để |
|---|---|
| `numpy` | tính toán mảng số |
| `pandas` | đọc Excel, bảng dữ liệu |
| `scipy.stats` | phân bố Poisson, Gauss, χ² |
| `matplotlib` | vẽ đồ thị |

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

## 2. Đọc file Excel thô

Đọc với `header=None` để pandas **không** coi hàng nào là tiêu đề. Khi đó chỉ số hàng/cột của pandas
= số hàng/cột Excel **trừ 1** (Excel bắt đầu từ 1, Python từ 0).

In [2]:
from du_lieu import thu_muc_bai          # tìm thư mục bài 'Đo Gamma' dù notebook mở ở đâu
FILE = thu_muc_bai() / "01_SoLieuGoc" / "nhóm 1.xlsx"
raw = pd.read_excel(FILE, header=None)
print(raw.shape)
raw.head(10)

(108, 8)


,0,1,2,3,4,5,6,7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Mẫu (Ra-226),NaN,NaN,NaN,Font,NaN,NaN
3,10s,STT,Số đếm 10s,NaN,NaN,Số đếm,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,STT,NaN,NaN
5,126,1,40,NaN,NaN,NaN,thời gian(10s),thời gian(100s)
6,115,2,26,NaN,NaN,1,10,45
7,109,3,33,NaN,NaN,2,9,NaN
8,128,4,34,NaN,NaN,3,7,NaN
9,134,5,37,NaN,NaN,4,9,NaN


Quan sát bảng: tiêu đề nằm ở hàng Excel 3–6, số liệu từ hàng 6–7 trở đi.

| Bộ | Vùng Excel | `iloc` tương ứng | Phép đo 100 s |
|---|---|---|---|
| S1 – mức thấp ("Font") | G7:G106 | `raw.iloc[6:106, 6]` | H7 → `raw.iloc[6, 7]` |
| S2 – Ra-226 trung bình | C6:C105 | `raw.iloc[5:105, 2]` | C108 → `raw.iloc[107, 2]` |
| S3 – Ra-226 cao | A6:A105 | `raw.iloc[5:105, 0]` | A108 → `raw.iloc[107, 0]` |

> 💡 **Kỹ thuật:** lát cắt `iloc[a:b]` lấy hàng `a` đến `b−1` → `iloc[5:105]` cho đúng 100 hàng.

In [3]:
x1 = raw.iloc[6:106, 6].astype(int).to_numpy()
x2 = raw.iloc[5:105, 2].astype(int).to_numpy()
x3 = raw.iloc[5:105, 0].astype(int).to_numpy()
N100 = {"S1": int(raw.iloc[6, 7]), "S2": int(raw.iloc[107, 2]), "S3": int(raw.iloc[107, 0])}

data = pd.DataFrame({"S1": x1, "S2": x2, "S3": x3}, index=pd.RangeIndex(1, 101, name="lan_do"))
print("Phép đo 100 s:", N100)
data.head()

Phép đo 100 s: {'S1': 45, 'S2': 437, 'S3': 1085}


,S1,S2,S3
lan_do,,,
1,10,40,126
2,9,26,115
3,7,33,109
4,9,34,128
5,5,37,134


## 3. Kiểm tra dữ liệu – luôn làm trước khi tính

Một lỗi lệch hàng (ví dụ lấy nhầm ô tiêu đề hoặc ô `AVERAGE`) sẽ làm sai toàn bộ kết quả.
Dùng `assert` để chương trình **tự báo lỗi** nếu dữ liệu không như mong đợi.

In [4]:
assert data.shape == (100, 3), "Phải có đúng 100 lần đo cho mỗi bộ"
assert not data.isna().any().any(), "Có ô trống!"
assert (data >= 0).all().all(), "Số đếm không thể âm"

# Đối chiếu với tổng/trung bình mà nhóm đã tính sẵn trong Excel (ô A106 = AVERAGE(A6:A105), C106)
print("Trung bình S3 tính lại:", data.S3.mean(), "| Excel A106:", raw.iloc[105, 0])
print("Trung bình S2 tính lại:", data.S2.mean(), "| Excel C106:", raw.iloc[105, 2])
data.describe().round(2)

Trung bình S3 tính lại: 112.19 | Excel A106: 112.19
Trung bình S2 tính lại: 41.22 | Excel C106: 41.22


,S1,S2,S3
count,100.00,100.00,100.00
mean,5.52,41.22,112.19
std,2.46,7.04,11.35
min,1.00,26.00,81.00
25%,4.00,36.00,105.00
50%,6.00,41.00,113.00
75%,7.00,47.00,120.00
max,12.00,58.00,139.00


## 4. Trung bình, độ lệch chuẩn – bẫy `ddof`

$$\bar x = \frac{1}{n}\sum x_i ,\qquad s = \sqrt{\frac{1}{n-1}\sum (x_i-\bar x)^2},\qquad u(\bar x)=\frac{s}{\sqrt n}$$

⚠️ `np.std(x)` mặc định chia cho $n$ (`ddof=0`), còn `pandas.Series.std()` và Excel `STDEV` chia cho $n-1$ (`ddof=1`).
Độ lệch chuẩn **mẫu** cần `ddof=1`.

In [5]:
x = data.S2.to_numpy()
print("np.std(x)          =", np.std(x))           # ddof=0  ✗
print("np.std(x, ddof=1)  =", np.std(x, ddof=1))   # ddof=1  ✓
print("data.S2.std()      =", data.S2.std())       # pandas mặc định ddof=1 ✓

np.std(x)          = 7.002256779067731
np.std(x, ddof=1)  = 7.03753285511824
data.S2.std()      = 7.03753285511824


In [6]:
n = len(data)
bang = pd.DataFrame({
    "x̄": data.mean(),
    "s": data.std(),                      # ddof=1
    "u(x̄)=s/√n": data.std() / np.sqrt(n),
    "√x̄ (Poisson)": np.sqrt(data.mean()),
    "R (s⁻¹)": data.mean() / 10,          # t = 10 s
})
bang.round(3)

,x̄,s,u(x̄)=s/√n,√x̄ (Poisson),R (s⁻¹)
S1,5.52,2.464,0.246,2.349,0.552
S2,41.22,7.038,0.704,6.420,4.122
S3,112.19,11.348,1.135,10.592,11.219


## 5. Phương sai có bằng trung bình không? – chỉ số phân tán $D$

Với phân bố Poisson $\sigma^2=\mu$ nên $D = s^2/\bar x \approx 1$.
Định lượng: nếu số liệu là Poisson thì $(n-1)D$ tuân theo phân bố $\chi^2$ với $n-1$ bậc tự do.
p-value hai phía nhỏ (< 0,05) nghĩa là phương sai **khác** Poisson một cách có ý nghĩa.

> 💡 **Kỹ thuật:** `stats.chi2.cdf` (xác suất ≤) và `stats.chi2.sf` (= 1 − cdf, chính xác hơn ở đuôi).

In [7]:
D = data.var() / data.mean()
chi2_D = (n - 1) * D
p_D = 2 * np.minimum(stats.chi2.cdf(chi2_D, n - 1), stats.chi2.sf(chi2_D, n - 1))
pd.DataFrame({"D = s²/x̄": D, "(n−1)D": chi2_D, "p-value": p_D}).round(3)

,D = s²/x̄,(n−1)D,p-value
S1,1.100,108.870,0.468
S2,1.202,118.951,0.168
S3,1.148,113.641,0.298


## 6. So sánh với phép đo 100 s

Quy phép đo 100 s về 10 s: $N_T\cdot t/T$, độ không đảm bảo $\sqrt{N_T}\cdot t/T$ (thống kê đếm).
Hiệu $\Delta$ được so với độ không đảm bảo tổng hợp:
$$z=\frac{\Delta}{\sqrt{u_1^2+u_2^2}}$$
$|z|\le 2$ → hai kết quả phù hợp.

In [8]:
t, T = 10, 100
rows = []
for bo in data:
    m, u = data[bo].mean(), data[bo].std() / np.sqrt(n)
    N = N100[bo]
    m100, u100 = N * t / T, np.sqrt(N) * t / T
    z = (m100 - m) / np.hypot(u, u100)        # np.hypot(a, b) = √(a² + b²)
    rows.append([bo, f"{m:.2f} ± {u:.2f}", f"{m100:.2f} ± {u100:.2f}", round(z, 2),
                 "Phù hợp" if abs(z) <= 2 else "Không phù hợp"])
pd.DataFrame(rows, columns=["Bộ", "x̄ (100×10 s)", "N_T/10", "z", "Kết luận"])

,Bộ,x̄ (100×10 s),N_T/10,z,Kết luận
0,S1,5.52 ± 0.25,4.50 ± 0.67,-1.43,Phù hợp
1,S2,41.22 ± 0.70,43.70 ± 2.09,1.12,Phù hợp
2,S3,112.19 ± 1.13,108.50 ± 3.29,-1.06,Phù hợp


## 7. Đóng gói thành hàm – tái sử dụng ở các notebook sau

Thay vì chép lại đoạn đọc file ở mọi nơi, ta viết **một hàm** trong file `du_lieu.py` (cùng thư mục) rồi `import`.
Lợi ích: sửa một chỗ, mọi notebook cập nhật; tránh lỗi chép nhầm chỉ số.

In [9]:
from du_lieu import doc_so_lieu
data2, N100_2 = doc_so_lieu()
assert data2.equals(data) and N100_2 == N100
print("Hàm doc_so_lieu() cho kết quả giống hệt ✓")

Hàm doc_so_lieu() cho kết quả giống hệt ✓


## 📝 Bài tập
1. Tính tốc độ đếm $R = \bar x / t$ và $u(R)$ cho cả 3 bộ (đơn vị s⁻¹).
2. Viết hàm `thong_ke(x)` trả về `dict` gồm `mean, s, u_mean, D, p_D`. Áp dụng cho từng cột bằng `data.apply(...)`.
3. Bộ 1 có thể là **phông nền**. Nếu trừ phông (tốc độ $R_1$) khỏi Bộ 2, tốc độ đếm thực của nguồn và độ không đảm bảo là bao nhiêu? *(Gợi ý: $u = \sqrt{u_1^2+u_2^2}$.)*